In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

In [ ]:
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

In [ ]:
L.seed_everything(42)
torch.set_float32_matmul_precision('medium')

Seed set to 42


In [ ]:
SEQ_LENGTH = 12             # Sequence history (e.g., 12 * 5 mins = 1 hour of history)
BATCH_SIZE = 64
HIDDEN_SIZE = 256
NUM_LAYERS = 4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 20

In [ ]:
class AlertSequenceDataset(Dataset):
    """
    Slices the datasets into sequences of a fixed history length.
    Maintains inputs as flat uint8 arrays in shared memory to save RAM, 
    preventing multiprocessing copies when num_workers > 0 on Windows.
    """
    def __init__(self, X_states, Y_states, time_features, seq_length=12):
        # We call .share_memory_() on these CPU tensors. This maps the data into 
        # system shared memory, allowing background workers to read from the 
        # same memory blocks without duplicating the tensors or pickling them.
        self.X_states = torch.tensor(X_states, dtype=torch.uint8).share_memory_()
        self.Y_states = torch.tensor(Y_states, dtype=torch.uint8).share_memory_()
        self.time_features = torch.tensor(time_features, dtype=torch.float32).share_memory_()
        self.seq_length = seq_length
        
    def __len__(self):
        # We can now utilize the entire timeline, predicting from step 0
        return len(self.X_states)
        
    def __getitem__(self, idx):
        if idx < self.seq_length - 1:
            # Initial phase: pad history by repeating/reusing the first available sample
            padding_len = self.seq_length - (idx + 1)
            
            # Slice available states and repeat the first state as padding
            avail_states = self.X_states[0 : idx + 1].to(torch.float32)
            pad_states = self.X_states[0].repeat(padding_len, 1).to(torch.float32)
            x_seq_states = torch.cat([pad_states, avail_states], dim=0)
            
            # Slice available time features and repeat the first time feature as padding
            avail_time = self.time_features[0 : idx + 1]
            pad_time = self.time_features[0].repeat(padding_len, 1)
            x_seq_time = torch.cat([pad_time, avail_time], dim=0)
        else:
            # Standard sequence slicing
            start_idx = idx - self.seq_length + 1
            x_seq_states = self.X_states[start_idx : idx + 1].to(torch.float32)
            x_seq_time = self.time_features[start_idx : idx + 1]
            
        # Concatenate along the feature dimension (states + cyclical time features)
        x_seq = torch.cat([x_seq_states, x_seq_time], dim=1)
        
        # Target corresponding exactly to this step (predicting t + 10 mins)
        y_target = self.Y_states[idx].to(torch.float32)
        
        return x_seq, y_target

In [ ]:
# %% [markdown]
# # Block 3: Lightning DataModule (Universal Configuration)

class AlertDataModule(L.LightningDataModule):
    """
    Handles memory-efficient loading of CSV files and parsing structures.
    Uses uint8 downcasting to minimize RAM usage.
    """
    def __init__(self, x_csv, y_csv, seq_length=12, batch_size=64, train_split=0.8, val_split=0.1):
        super().__init__()
        self.x_csv = x_csv
        self.y_csv = y_csv
        self.seq_length = seq_length
        self.batch_size = batch_size
        self.train_split = train_split
        self.val_split = val_split
        
        self.input_size = None
        self.output_size = None
        
    def setup(self, stage=None):
        # 1. Read first row to determine column headers
        print(" -> Initializing CSV parsing metadata...")
        x_headers = pd.read_csv(self.x_csv, nrows=1).columns
        y_headers = pd.read_csv(self.y_csv, nrows=1).columns
        
        # Explicitly instruct pandas to load all state columns as uint8 (1 byte per cell)
        x_dtypes = {col: 'uint8' for col in x_headers if col != 'timestamp'}
        y_dtypes = {col: 'uint8' for col in y_headers if col != 'target_timestamp'}
        
        # 2. Read full CSVs memory-efficiently
        print(" -> Loading X_dataset.csv (Optimized uint8 format)...")
        df_x = pd.read_csv(self.x_csv, dtype=x_dtypes)
        print(" -> Loading Y_dataset.csv (Optimized uint8 format)...")
        df_y = pd.read_csv(self.y_csv, dtype=y_dtypes)
        
        # 3. Parse timestamps and extract cyclical time coordinates
        print(" -> Precomputing temporal coordinates...")
        timestamps = pd.to_datetime(df_x['timestamp'])
        hours = timestamps.dt.hour.values
        day_of_weeks = timestamps.dt.dayofweek.values
        
        hour_sin = np.sin(2 * np.pi * hours / 24.0)
        hour_cos = np.cos(2 * np.pi * hours / 24.0)
        day_sin = np.sin(2 * np.pi * day_of_weeks / 7.0)
        day_cos = np.cos(2 * np.pi * day_of_weeks / 7.0)
        
        # Keep time features as float32 in memory (shape: num_samples, 4) -> very small (~1.6 MB)
        time_features = np.stack([hour_sin, hour_cos, day_sin, day_cos], axis=1).astype(np.float32)
        
        # 4. Extract state arrays as raw uint8 numpy arrays
        X_states = df_x.drop(columns=['timestamp']).values
        Y_states = df_y.drop(columns=['target_timestamp']).values
        
        self.output_size = Y_states.shape[1]
        self.input_size = X_states.shape[1] + 4
        
        # 5. Build sequence dataset using lazy processing
        full_dataset = AlertSequenceDataset(X_states, Y_states, time_features, self.seq_length)
        
        # 6. Calculate split lengths
        num_samples = len(full_dataset)
        train_len = int(self.train_split * num_samples)
        val_len = int(self.val_split * num_samples)
        test_len = num_samples - train_len - val_len
        
        self.train_dataset, self.val_dataset, self.test_dataset = random_split(
            full_dataset, [train_len, val_len, test_len],
            generator=torch.Generator().manual_seed(42)
        )
        
    def train_dataloader(self):
        return DataLoader(
            self.train_dataset, 
            batch_size=self.batch_size, 
            shuffle=True, 
            num_workers=0,  # Set to 0 to prevent multiprocessing freezes on Windows
            pin_memory=True
        )
        
    def val_dataloader(self):
        return DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size, 
            shuffle=False, 
            num_workers=0,  # Set to 0 to prevent multiprocessing freezes on Windows
            pin_memory=True
        )
        
    def test_dataloader(self):
        return DataLoader(
            self.test_dataset, 
            batch_size=self.batch_size, 
            shuffle=False, 
            num_workers=0,  # Set to 0 to prevent multiprocessing freezes on Windows
            pin_memory=True
        )

In [ ]:

class AlertPredictionModel(L.LightningModule):
    """
    Recurrent sequence architecture using multi-layer LSTMs.
    Takes cyclical temporal vectors alongside binary states as input.
    """
    def __init__(self, input_size, hidden_size, output_size, num_layers=2, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, output_size)
        self.loss_fn = nn.BCEWithLogitsLoss()
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_step_out = lstm_out[:, -1, :]
        logits = self.fc(last_step_out)
        return logits
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss
        
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log("val_loss", loss, on_epoch=True, prog_bar=True)
        return loss
        
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)



In [ ]:
if __name__ == "__main__":
    # Define file paths
    if os.path.exists("DataPreprocess"):
        X_DATA_CSV = os.path.join("DataPreprocess", "FinalData", "X_dataset.csv")
        Y_DATA_CSV = os.path.join("DataPreprocess", "FinalData", "Y_dataset.csv")
        CHECKPOINT_DIR = os.path.join("AlertModel", "checkpoints")
    else:
        X_DATA_CSV = os.path.join("..", "DataPreprocess", "FinalData", "X_dataset.csv")
        Y_DATA_CSV = os.path.join("..", "DataPreprocess", "FinalData", "Y_dataset.csv")
        CHECKPOINT_DIR = "checkpoints"

    print("[Pipeline Status] Checking for dataset files...")
    if not os.path.exists(X_DATA_CSV) or not os.path.exists(Y_DATA_CSV):
        print(f"[Error] Missing '{X_DATA_CSV}' or '{Y_DATA_CSV}'. Please run 'generate_dataset.py' first.")
    else:
        print("[Pipeline Status] Dataset files located. Setting up DataModule...")
        data_module = AlertDataModule(
            x_csv=X_DATA_CSV,
            y_csv=Y_DATA_CSV,
            seq_length=SEQ_LENGTH,
            batch_size=BATCH_SIZE
        )
        data_module.setup()
        
        print(f"[Pipeline Status] Datasets prepared and split successfully:")
        print(f"  - Training samples:   {len(data_module.train_dataset)}")
        print(f"  - Validation samples: {len(data_module.val_dataset)}")
        print(f"  - Testing samples:    {len(data_module.test_dataset)}")
        
        print(f"[Pipeline Status] Initializing LSTM model...")
        print(f"  - Input dimensions (Units + 4 Time Coordinates): {data_module.input_size}")
        print(f"  - Output dimensions (Target Units):             {data_module.output_size}")
        print(f"  - Hidden dimension size:                        {HIDDEN_SIZE}")
        print(f"  - LSTM Recurrent layers:                        {NUM_LAYERS}")
        
        model = AlertPredictionModel(
            input_size=data_module.input_size,
            hidden_size=HIDDEN_SIZE,
            output_size=data_module.output_size,
            num_layers=NUM_LAYERS,
            lr=LEARNING_RATE
        )
        
        print("[Pipeline Status] Registering Early Stopping and Checkpoint callbacks...")
        checkpoint_callback = ModelCheckpoint(
            monitor="val_loss",
            dirpath=CHECKPOINT_DIR,
            filename="best-lstm-alert-model",
            save_top_k=1,
            mode="min"
        )
        
        early_stopping_callback = EarlyStopping(
            monitor="val_loss",
            patience=3,
            mode="min"
        )
        
        print("[Pipeline Status] Initializing L.Trainer engine...")
        trainer = L.Trainer(
            max_epochs=MAX_EPOCHS,
            callbacks=[checkpoint_callback, early_stopping_callback],
            accelerator="auto", 
            devices="auto"
        )
        
        print("\n=== [Pipeline Status] Beginning model training loop ===")
        trainer.fit(model, datamodule=data_module)
        print("=== [Pipeline Status] Model training phase successfully completed ===")
        
        best_path = checkpoint_callback.best_model_path
        if best_path:
            print(f"[Pipeline Status] Best model checkpoint saved to: '{best_path}'")

[Pipeline Status] Checking for dataset files...
[Pipeline Status] Dataset files located. Setting up DataModule...
 -> Initializing CSV parsing metadata...
 -> Loading X_dataset.csv (Optimized uint8 format)...
 -> Loading Y_dataset.csv (Optimized uint8 format)...
 -> Precomputing temporal coordinates...
[Pipeline Status] Datasets prepared and split successfully:
  - Training samples:   358904
  - Validation samples: 44863
  - Testing samples:    44864
[Pipeline Status] Initializing LSTM model...
  - Input dimensions (Units + 4 Time Coordinates): 1626
  - Output dimensions (Target Units):             1622
  - Hidden dimension size:                        256
  - LSTM Recurrent layers:                        4
[Pipeline Status] Registering Early Stopping and Checkpoint callbacks...
[Pipeline Status] Initializing L.Trainer engine...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores



=== [Pipeline Status] Beginning model training loop ===
 -> Initializing CSV parsing metadata...
 -> Loading X_dataset.csv (Optimized uint8 format)...
 -> Loading Y_dataset.csv (Optimized uint8 format)...
 -> Precomputing temporal coordinates...


c:\Users\yablo\.virtualenvs\ml_env\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\yablo\OneDrive\Desktop\CodeStudio\AlertsPredict_simple\AlertModel\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type              | Params | Mode  | FLOPs
--------------------------------------------------------------
0 | lstm    | LSTM              | 3.5 M  | train | 0    
1 | fc      | Linear            | 416 K  | train | 0    
2 | loss_fn | BCEWithLogitsLoss | 0      | train | 0    
--------------------------------------------------------------
3.9 M     Trainable params
0         Non-trainable params
3.9 M     Total params
15.700    Total estimated model params size (MB)
3         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\yablo\.virtualenvs\ml_env\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
